# 04 — Stat-only likelihood scans

Use the [NSBI LHC toolkit](https://github.com/iris-hep/nsbi-lhc-toolkit/) for an **unbinned, extended likelihood**: both the event yield and the density-ratio information enter the fit. The only fitted parameter is $\mu$; there are no nuisance parameters.

| Process | Scaling |
|---|---|
| SBI | $\sqrt{\mu}$ |
| S | $\mu-\sqrt{\mu}$ |
| B | $1-\sqrt{\mu}$ |
| qqZZ | $1$ |

The reference is still **S + EWSBI10**, with signed `weight_kFact` weights. Use the **full reference first**, then **2,048**, **16,384**, **131,072**, and **1,048,576** randomly selected events. One seeded permutation makes the smaller subsets nested, with no replacement.

We use **SM Asimov data at $\mu=1$** on each reference sample. All four physical process yields come from their full native samples and stay fixed for every curve. Reducing the reference size changes the Monte Carlo approximation, **not the luminosity or expected event yield**. These are expected likelihood curves, not fits to observed collision data.

## Setup

Use a **CPU runtime**. The toolkit is pinned to an inspected commit, and the Matplotlib/mplhep versions match notebooks 01–03. If setup reports a stale import, choose **Runtime → Restart session**, then **Run all**.

`utils.py` supplies the existing HDF5 reader. `utils_likelihood.py` formats the toolkit workspace and maps the physical POI to its normalization factors. It contains no likelihood implementation. The toolkit handles the likelihood, fit, scan, and curve drawing.

In [1]:
%pip -q install "matplotlib==3.10.0" "mplhep==0.4.1" tables "jax[cpu]==0.5.3" "iminuit==2.33.0" "scikit-learn==1.9.1" "git+https://github.com/iris-hep/nsbi-lhc-toolkit.git@fc09848fc6540fd32310faebbe9db6eea7ecd17b"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 18.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
flax 0.11.2 requires jax>=0.6.0, but you have jax 0.5.3 which is incompatible.
orbax-checkpoint 0.12.7 requires jax>=0.6.0, but you have jax 0.5.3 which is incompatible.


In [2]:
import sys
import os
from importlib.metadata import version

packages = {name: name for name in (
    "numpy", "pandas", "matplotlib", "mplhep", "tables", "jax", "jaxlib", "iminuit", "scipy"
)}
packages["sklearn"] = "scikit-learn"
for module_name, package in packages.items():
    loaded = sys.modules.get(module_name)
    if loaded is not None and loaded.__version__ != version(package):
        raise RuntimeError(
            f"{package} changed on disk while it was loaded. "
            "Choose Runtime > Restart session, then Run all."
        )
os.environ["JAX_PLATFORMS"] = "cpu"

import gc
import json
import importlib
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
import jax
from IPython.display import display
from nsbi_common_utils import models, inference

try:
    from google.colab import drive
except ImportError:
    import utils as offshell_utils
    import utils_likelihood
else:
    drive.mount("/content/drive")
    repo_url = "https://raw.githubusercontent.com/rafaellopesdesa/offshell_reweighting/main"
    urlretrieve(f"{repo_url}/utils.py", "/content/offshell_utils.py")
    urlretrieve(f"{repo_url}/utils_likelihood.py", "/content/utils_likelihood.py")
    import offshell_utils
    import utils_likelihood

importlib.invalidate_caches()
importlib.reload(offshell_utils)
importlib.reload(utils_likelihood)
hep.style.use("ATLAS")
offshell_utils.check_plot_exports()
print("Toolkit commit:", utils_likelihood.TOOLKIT_COMMIT)
print("JAX devices:", jax.devices())
print("PNG and PDF export checks passed.")


Mounted at /content/drive
Toolkit commit: fc09848fc6540fd32310faebbe9db6eea7ecd17b


ERROR:2026-10-20 17:16:18,082:jax._src.xla_bridge:626: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda13.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/jax/_src/xla_bridge.py", line 624, in discover_pjrt_plugins
    plugin_module.initialize()
    ~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/jax_plugins/xla_cuda13/__init__.py", line 378, in initialize
    xla_client.register_custom_type_handler(
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module 'jaxlib.xla_client' has no attribute 'register_custom_type_handler'. Did you mean: 'register_custom_type_id_handler'?
ERROR:jax._src.xla_bridge:Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda13.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/jax/_src/xla_bridge.py", line 624, in discover_pjrt_plugins
    plugin_module.initialize()
    ~~~~~~~~~~~~~~~

JAX devices: [CpuDevice(id=0)]
PNG and PDF export checks passed.


In [3]:
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/HZZ_OffShell")
DATA_PATH = PROJECT_DIR / "dataset_wDR_mean_origIndex.h5"
RESULT_DIR = PROJECT_DIR / "likelihoods" / "04_stat_only"
PLOT_DIR = PROJECT_DIR / "plots" / "04_likelihood_scans"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

PROCESSES = ["SBI", "S", "B", "qqZZ"]
SUBSET_SIZES = [2048, 16384, 131072, 1048576]
RANDOM_SEED = 42
SCAN_RANGE = (0.0, 5.0)
SCAN_POINTS = 101  # Step 0.05; includes the Asimov truth mu=1 exactly.


## Load the reference and fix the physical yields

Keep the same finite-value selection as notebooks 01–03: finite `weight_kFact`, `NN_disc_New`, and `m4l_fsr`. Values outside the old histogram ranges remain included. The likelihood uses the stored ratios directly, without binning those variables.

Ratios are checked only on reference events. All signed weights are retained. The native S sample also belongs to the reference; that overlap is intentional.

In [4]:
data = offshell_utils.load_columns(DATA_PATH, [
    "Type", "weight_kFact", "NN_disc_New", "m4l_fsr",
    "r_SBI", "r_S", "r_B", "r_qqZZ",
])
event_type = data["Type"]
weights = np.asarray(data["weight_kFact"], dtype=np.float64)
finite = (np.isfinite(weights) & np.isfinite(data["NN_disc_New"])
          & np.isfinite(data["m4l_fsr"]))
reference = np.isin(event_type, ["S", "EWSBI10"]) & finite
reference_rows = np.flatnonzero(reference)
w_ref = weights[reference]
r_ref = {name: np.asarray(data[f"r_{name}"][reference], dtype=np.float64)
         for name in PROCESSES}
for name, ratios in r_ref.items():
    if not np.isfinite(ratios).all():
        raise ValueError(f"Nonfinite r_{name} on reference events; inspect the input.")

native_yields = {}
yield_rows = []
for name in ["Reference"] + PROCESSES:
    original = (np.isin(event_type, ["S", "EWSBI10"]) if name == "Reference"
                else event_type == name)
    mask = original & finite
    total = float(weights[mask].sum())
    native_yields[name] = total
    yield_rows.append({"sample": name, "events": int(mask.sum()), "yield": total,
                       "excluded_nonfinite": int(np.count_nonzero(original & ~finite)),
                       "negative_weights": int(np.count_nonzero(weights[mask] < 0))})
yield_table = pd.DataFrame(yield_rows).set_index("sample")
display(yield_table)
yield_table.to_csv(RESULT_DIR / "native_yields.csv")
if any(not np.isfinite(value) or value <= 0 for value in native_yields.values()):
    raise ValueError("Each physical sample needs a finite, positive total signed yield.")
if max(SUBSET_SIZES) > len(w_ref):
    raise ValueError(f"Only {len(w_ref):,} selected reference events; cannot draw all sizes.")

# Keep only the reference columns in memory for the scan.
del data, event_type, weights, finite, reference, original, mask, ratios
gc.collect()


,events,yield,excluded_nonfinite,negative_weights
sample,,,,
Reference,3110759,91.773434,0,1333
SBI,1673988,359.285953,0,122
S,2222114,36.855833,0,415
B,1011799,368.407214,0,85
qqZZ,2643586,1651.317870,0,488161


11619

## Same-sample normalization for every reference size

For the selected reference events, calculate
$$
u_i = \frac{w_i}{\sum_{k\in\mathrm{subset}} w_k},\qquad
Z_j = \sum_{i\in\mathrm{subset}} u_i r_j(x_i),\qquad
\widehat r_j(x_i)=\frac{r_j(x_i)}{Z_j}.
$$
Do this separately for **SBI, S, B, and qqZZ**, including the full-reference case. Each corrected weighted mean must equal one. Never reuse the full-sample Z factors for a smaller sample.

At $\mu=1$, only SBI and qqZZ contribute. Supply the toolkit with the Asimov event weights
$$
a_i = u_i\left(\lambda_{\mathrm{SBI}}\widehat r_{\mathrm{SBI}}(x_i)
+\lambda_{\mathrm{qqZZ}}\widehat r_{\mathrm{qqZZ}}(x_i)\right).
$$
Their sum is always $\lambda_{\mathrm{SBI}}+\lambda_{\mathrm{qqZZ}}$. Negative MC weights remain signed. The resulting curves probe reference-sample convergence; there are no MC-statistical or ratio-training nuisance parameters.

In [5]:
permutation = np.random.default_rng(RANDOM_SEED).permutation(len(w_ref))
cases = [("full", len(w_ref))] + [(f"n_{n}", n) for n in SUBSET_SIZES]
workspace_paths = {}
normalization_rows = []
expected_yield = native_yields["SBI"] + native_yields["qqZZ"]

for label, n in cases:
    selected = np.arange(n) if label == "full" else permutation[:n]
    sample_dir = RESULT_DIR / label
    sample_dir.mkdir(parents=True, exist_ok=True)
    selected_weight_sum = float(w_ref[selected].sum())
    if not np.isfinite(selected_weight_sum) or selected_weight_sum == 0:
        raise ValueError(f"{label}: reference weights cannot be normalized.")
    u = w_ref[selected] / selected_weight_sum
    np.testing.assert_allclose(u.sum(), 1.0, rtol=1e-10)

    z = {}
    corrected = {}
    for name in PROCESSES:
        r = r_ref[name][selected]
        z[name] = float(np.sum(u * r))
        if not np.isfinite(z[name]) or z[name] == 0:
            raise ValueError(f"{label}: invalid Z_{name} = {z[name]}")
        corrected[name] = r / z[name]
        mean_after = float(np.sum(u * corrected[name]))
        np.testing.assert_allclose(mean_after, 1.0, rtol=1e-10)
        np.save(sample_dir / f"r_{name}.npy", corrected[name])
        normalization_rows.append({"sample": label, "events": n, "process": name,
                                   "Z": z[name], "corrected_mean": mean_after})

    asimov = u * (native_yields["SBI"] * corrected["SBI"]
                  + native_yields["qqZZ"] * corrected["qqZZ"])
    np.testing.assert_allclose(asimov.sum(), expected_yield, rtol=1e-10)
    np.save(sample_dir / "asimov_weights.npy", asimov)
    np.save(sample_dir / "reference_weights_normalized.npy", u)
    np.save(sample_dir / "event_rows.npy", reference_rows[selected])
    workspace_paths[label] = utils_likelihood.write_workspace(
        sample_dir, native_yields,
        metadata={"reference_events": n, "reference_types": ["S", "EWSBI10"],
                  "random_seed": RANDOM_SEED, "Z": z,
                  "selected_reference_weight_sum": selected_weight_sum,
                  "asimov_yield": float(asimov.sum()),
                  "negative_reference_weights": int(np.count_nonzero(u < 0)),
                  "input_file": str(DATA_PATH.resolve())},
    )
    print(f"{label}: {n:,} reference events; Asimov yield = {asimov.sum():.6f}")

normalization_table = pd.DataFrame(normalization_rows)
display(normalization_table.set_index(["sample", "process"]))
normalization_table.to_csv(RESULT_DIR / "normalization_checks.csv", index=False)
del corrected, r, asimov, u, selected
gc.collect()


full: 3,110,759 reference events; Asimov yield = 2010.603823
n_2048: 2,048 reference events; Asimov yield = 2010.603823
n_16384: 16,384 reference events; Asimov yield = 2010.603823
n_131072: 131,072 reference events; Asimov yield = 2010.603823
n_1048576: 1,048,576 reference events; Asimov yield = 2010.603823


events         Z  corrected_mean
sample    process                                   
full      SBI      3110759  1.000214             1.0
          S        3110759  1.000264             1.0
          B        3110759  1.000296             1.0
          qqZZ     3110759  1.000877             1.0
n_2048    SBI         2048  1.018697             1.0
          S           2048  0.971859             1.0
          B           2048  1.018666             1.0
          qqZZ        2048  1.029697             1.0
n_16384   SBI        16384  1.011555             1.0
          S          16384  0.996665             1.0
          B          16384  1.011277             1.0
          qqZZ       16384  1.010079             1.0
n_131072  SBI       131072  1.011932             1.0
          S         131072  1.001549             1.0
          B         131072  1.012163             1.0
          qqZZ      131072  1.007264             1.0
n_1048576 SBI      1048576  1.004186             1.0
          S        1048576  1.000237             1.0
          B        1048576  1.004166             1.0
          qqZZ     1048576  1.004977             1.0

0

## Fit and scan with the toolkit

Each `workspace.json` stores the four full process yields, paths to the corrected ratios, and the Asimov weights. Its metadata records the subset, seed, Z factors, and physical scaling formula.

The pinned toolkit supports multiplicative normfactors. The small `mu_likelihood` adapter ties its three internal factors to $(\sqrt\mu,\mu-\sqrt\mu,1-\sqrt\mu)$; **only one physical parameter is fitted**. qqZZ has no modifier. Negative S or B coefficients are allowed; the combined intensity must remain physical. No densities or weights are clipped.

`models.sbi_parametric_model` computes the extended likelihood. `inference.inference` fits and scans it. The model has no nuisance parameters, so the ordinary scan is already **stat-only**. The toolkit returns $t_\mu=-2\Delta\log L$ with the scan minimum subtracted. The grid contains the Asimov truth $\mu=1$ exactly.

In [6]:
scan_grid = np.linspace(*SCAN_RANGE, SCAN_POINTS)
if SCAN_RANGE[0] < 0 or not np.any(np.isclose(scan_grid, 1.0, atol=1e-12, rtol=0)):
    raise ValueError("Use a nonnegative scan grid containing mu=1 exactly.")

scans = {}
fit_rows = []
for label, n in cases:
    print(f"\n--- {label}: {n:,} reference events ---")
    workspace = json.loads(workspace_paths[label].read_text())
    model = models.sbi_parametric_model(workspace, measurement_to_fit="stat_only")
    nll_mu = utils_likelihood.mu_likelihood(model)
    if not np.isfinite(nll_mu([1.0])):
        raise ValueError(f"{label}: nonphysical total intensity at the Asimov point.")

    fitter = inference.inference(
        model_nll=nll_mu, initial_values=[1.0], list_parameters=["mu"],
        num_unconstrained_params=1,
    )
    fitter.perform_fit()
    mu, t_mu = fitter.perform_profile_scan(
        parameter_name="mu", bound_range=SCAN_RANGE, size=SCAN_POINTS,
    )
    mu, t_mu = np.asarray(mu), np.asarray(t_mu)
    if not np.isfinite(t_mu).all():
        raise ValueError(
            f"{label}: nonfinite likelihood scan. Inspect the signed ratios and "
            "combined intensity over SCAN_RANGE; no invalid points were clipped."
        )
    scans[label] = (mu, t_mu)
    pd.DataFrame({"mu": mu, "t_mu": t_mu}).to_csv(
        workspace_paths[label].parent / "scan.csv", index=False,
    )
    fit_rows.append({"sample": label, "reference_events": n,
                     "mu_hat": float(fitter.pulls_global_fit[0]),
                     "grid_minimum_mu": float(mu[np.argmin(t_mu)]),
                     "asimov_yield": workspace["metadata"]["asimov_yield"]})
    del fitter, nll_mu, model
    jax.clear_caches()  # Release the large compiled model before the next subset.
    gc.collect()

fit_table = pd.DataFrame(fit_rows).set_index("sample")
display(fit_table)
fit_table.to_csv(RESULT_DIR / "fit_summary.csv")



--- full: 3,110,759 reference events ---
fit: 
 ┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = -3.127e+04                 │              Nfcn = 20               │
│ EDM = 1.06e-15 (Goal: 0.0002)    │            time = 0.7 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬───

/usr/local/lib/python3.13/dist-packages/nsbi_common_utils/inference.py:303: IMinuitWarning: MIGRAD fails to converge for mu=1.9000000000000001
  scan_points, NLL_value, _ = m.mnprofile(parameter_name,
/usr/local/lib/python3.13/dist-packages/nsbi_common_utils/inference.py:303: IMinuitWarning: MIGRAD fails to converge for mu=1.9500000000000002
  scan_points, NLL_value, _ = m.mnprofile(parameter_name,
/usr/local/lib/python3.13/dist-packages/nsbi_common_utils/inference.py:303: IMinuitWarning: MIGRAD fails to converge for mu=2.0
  scan_points, NLL_value, _ = m.mnprofile(parameter_name,
/usr/local/lib/python3.13/dist-packages/nsbi_common_utils/inference.py:303: IMinuitWarning: MIGRAD fails to converge for mu=2.0500000000000003
  scan_points, NLL_value, _ = m.mnprofile(parameter_name,
/usr/local/lib/python3.13/dist-packages/nsbi_common_utils/inference.py:303: IMinuitWarning: MIGRAD fails to converge for mu=2.1
  scan_points, NLL_value, _ = m.mnprofile(parameter_name,
/usr/local/lib/python3.13

ValueError: full: nonfinite likelihood scan. Inspect the signed ratios and combined intensity over SCAN_RANGE; no invalid points were clipped.

## Compare the five likelihood curves

The toolkit draws every curve and the reference lines at 1, 4, and 9. ATLAS styling comes from mplhep. These are likelihood axes, not histogram ratio panels.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))
labels = [f"Full reference ({len(w_ref):,})"] + [f"{n:,} reference events" for n in SUBSET_SIZES]
inference.plot_NLL_scans(
    parameter_label=r"$\mu$",
    list_scan_points=[scans[label][0] for label, _ in cases],
    list_nll_values=[scans[label][1] for label, _ in cases],
    list_labels=labels,
    list_linestyles=["solid", "dotted", "dashed", "dashdot", (0, (5, 2))],
    list_colors=["black", "#D55E00", "#0072B2", "#009E73", "#CC79A7"],
    ax=ax,
)
ax.set_xlim(*SCAN_RANGE)
ax.set_ylabel(r"$t_\mu=-2\Delta\log L$")
ax.text(0.02, 1.06, "ATLAS", transform=ax.transAxes,
        fontsize=17, fontweight="bold", fontstyle="italic")
ax.text(0.19, 1.06, "Simulation Internal", transform=ax.transAxes, fontsize=15)
ax.text(0.02, 1.01, r"Stat-only · Asimov $\mu=1$ · Same-sample normalization",
        transform=ax.transAxes, fontsize=12)
ax.legend(loc="best", fontsize=11, frameon=False)
fig.tight_layout()
for extension in ("png", "pdf"):
    fig.savefig(PLOT_DIR / f"reference_size_comparison.{extension}", dpi=180, bbox_inches="tight")
plt.show()
print("Workspaces, arrays, Z checks, and scan tables:", RESULT_DIR)
print("Likelihood figures:", PLOT_DIR)


The Z table should have `corrected_mean = 1` for all 20 entries. All five Asimov yields should agree, and the fitted/grid minima should lie at or close to `mu = 1`. Differences between the curves then show the effect of reference-sample size for this seeded draw.

The JSON workspaces and their `.npy` inputs stay in Drive. To reopen one, load its JSON, create `models.sbi_parametric_model(workspace, "stat_only")`, and apply `utils_likelihood.mu_likelihood(model)` before fitting the single physical parameter. The three internal normalization factors must stay tied to mu.